# 01. Build the RUL dataset from the raw ISU-ILCC records

Turns the raw cycling records of the ISU-ILCC Battery Aging Dataset (about 11 GB) into one small table, `rul_steps.csv`, with one row per weekly capacity test of every cell. The table is already included in `data/`, so this notebook only needs to be run to reproduce it.

Run in Google Colab with `RPT_json.zip`, `Cycling_json.zip` and `Valid_cells.csv` in the top level of Google Drive.

## Setup
Mount Google Drive and set paths and constants.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, glob, json, gc, zipfile
import numpy as np
import pandas as pd

ROOT = '/content/drive/MyDrive'
VALID_CELLS = f'{ROOT}/Valid_cells.csv'
OUTPUT = f'{ROOT}/rul_steps.csv'

RATED_AH = 0.250           # rated capacity
EOL_AH = 0.8 * RATED_AH    # end of life: 80% of rated capacity

## Reading the raw files
Each cell has an RPT file (weekly capacity tests) and a cycling file (every charge-discharge cycle). Both are JSON wrapped in a JSON string, and missing records are stored as `'[]'`.

In [ ]:
def json_paths(name):
    # Cell name -> JSON path. A ZIP, if present, is extracted inside Colab (faster than reading from Drive).
    zip_path, out = f'{ROOT}/{name}.zip', f'/content/data/{name}'
    if not os.path.exists(out) and os.path.exists(zip_path):
        print(f'Extracting {name}.zip ...')
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(out)
    for base in [out, f'{ROOT}/{name}']:
        paths = sorted(glob.glob(f'{base}/**/*.json', recursive=True))
        if paths:
            return {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
    return {}

def load_json(path):
    with open(path) as f:
        return json.loads(json.load(f))   # decode twice

def to_time(values):
    clean = pd.Series([v if isinstance(v, str) and v != '[]' else None for v in values], dtype=object)
    return pd.to_datetime(clean, format='ISO8601', errors='coerce')

def to_num(values):
    return pd.Series([v if isinstance(v, (int, float)) else np.nan for v in values], dtype=float)

def first_currents(cyc, n_cycles=30):
    # (max charge current, median discharge current) over the first n_cycles cycles
    chg, dis = [], []
    for i_chg, i_dis in zip(cyc['QV_charge']['I'][:n_cycles], cyc['QV_discharge']['I'][:n_cycles]):
        if len(i_chg):
            chg.append(np.max(np.abs(i_chg)))
        if len(i_dis):
            dis.append(np.median(np.abs(i_dis)))
    return (np.median(chg) if chg else np.nan), (np.median(dis) if dis else np.nan)

## One row per weekly test
For every weekly test before the end of life:

- `capacity`: full capacity measured at that test (Ah)
- `efc_so_far`: equivalent full cycles (EFC) used before that test = cumulative discharged Ah / 0.25
- `rul_efc`: EFC remaining until capacity reaches 80% of rated capacity (the prediction target)
- `c_chg`, `c_dis`, `dod`: charge rate, discharge rate and depth of discharge, measured from the first cycles

The end of life is interpolated between the two weekly tests around the 80% threshold.

In [ ]:
def rul_rows(rpt, cyc):
    # Weekly tests: capacity and start time (keep tests that have both)
    cap, t_rpt = to_num(rpt['capacity_discharge_C_5']), to_time(rpt['start_stop_time']['start'])
    n = min(len(cap), len(t_rpt))
    ok = (cap[:n].notna() & t_rpt[:n].notna()).values
    cap, t_rpt = cap[:n].values[ok], t_rpt[:n].values[ok]

    # Cycles: discharged capacity and discharge start time, sorted by time
    q_all, t_cyc = to_num(cyc['capacity_discharge']), to_time(cyc['time_series_discharge']['start'])
    n = min(len(q_all), len(t_cyc))
    # A missing capacity is filled with the median of the cell's earlier cycles only, so no later
    # information enters the EFC used so far (the counts printed below show none were missing).
    q = q_all[:n]
    q, t_cyc = q.fillna(q.expanding().median().shift(1)).fillna(q.dropna().iloc[:10].median()), t_cyc[:n]
    keep = t_cyc.notna().values
    q, t_cyc = q.values[keep], t_cyc.values[keep]
    order = np.argsort(t_cyc)
    q, t_cyc = q[order], t_cyc[order]

    n_before = np.searchsorted(t_cyc, t_rpt)                                 # cycles started before each test
    efc_before = np.concatenate([[0.0], np.cumsum(q)])[n_before] / RATED_AH  # EFC used before each test

    below = np.where(cap < EOL_AH)[0]
    if not len(q) or not len(below) or below[0] == 0:
        return []                                  # never reached end of life, or no usable record
    i = below[0]
    frac = (cap[i-1] - EOL_AH) / (cap[i-1] - cap[i])
    efc_total = efc_before[i-1] + frac * (efc_before[i] - efc_before[i-1])   # EFC at the 80% crossing

    i_chg, i_dis = first_currents(cyc)
    first_q = q_all.dropna().values[:10]
    conditions = {'c_chg': i_chg / RATED_AH, 'c_dis': i_dis / RATED_AH,
                  'dod': float(np.median(first_q) / cap[0])}
    return [{'step': k, 'capacity': float(cap[k]), 'efc_so_far': float(efc_before[k]),
             'rul_efc': float(efc_total - efc_before[k]), 'efc_total': float(efc_total), **conditions}
            for k in range(i)]                     # tests before the end of life

valid = pd.read_csv(VALID_CELLS, header=None, dtype=str).values.ravel()
valid = [v.strip() for v in valid if isinstance(v, str) and re.fullmatch(r'G\d+C\d+', v.strip())]
rpt_paths, cyc_paths = json_paths('RPT_json'), json_paths('Cycling_json')
todo = [c for c in valid if c in rpt_paths and c in cyc_paths]
print('Cells to read:', len(todo))

rows = []
n_filled = n_no_time = 0
for n_done, cell in enumerate(todo, 1):
    rpt, cyc = load_json(rpt_paths[cell]), load_json(cyc_paths[cell])
    q_raw, t_raw = to_num(cyc['capacity_discharge']), to_time(cyc['time_series_discharge']['start'])
    n = min(len(q_raw), len(t_raw))
    n_filled += int((q_raw[:n].isna() & t_raw[:n].notna()).sum())       # capacities rul_rows has to fill
    n_no_time += int(t_raw[:n].isna().sum())                             # cycles left out of the EFC count
    group = int(re.fullmatch(r'G(\d+)C(\d+)', cell).group(1))     # cells in a group share the same conditions
    rows += [{'cell': cell, 'group': group, **r} for r in rul_rows(rpt, cyc)]
    del rpt, cyc
    gc.collect()
    if n_done % 25 == 0:
        print(f'  {n_done}/{len(todo)} done')

print('Cycles with a missing capacity (filled):', n_filled, '| cycles without a start time (not counted):', n_no_time)
steps = pd.DataFrame(rows)
steps.to_csv(OUTPUT, index=False)
print('Saved', OUTPUT, '|', steps['cell'].nunique(), 'cells,', len(steps), 'rows')

## Voltage-curve features (optional)
For section 13 of `02_train_models.ipynb`. From the C/5 discharge curve of every weekly test: the change since the first test, ΔQ(V) on 3.40-4.05 V (mean, minimum, log variance), the C/2-to-C/5 capacity ratio, the voltage and height of the largest dQ/dV peak, and the capacity discharged down to 3.40 V. Writes `dqv_features.csv` to My Drive (copy it to `data/`). Takes about 3 minutes.

In [ ]:
import time

DQV_OUTPUT = f'{ROOT}/dqv_features.csv'
V_LO, V_HI = 3.40, 4.05                           # voltage window shared by all C/5 discharge curves (V)
V_GRID = np.linspace(V_LO, V_HI, 400)
UNIT_FACTORS = [1.0, 1e-3, 1 / 3600, 1 / 3.6e6]   # Q in Ah, mAh, A·s or mA·s -> Ah


def field(d, name):
    for k in d:                                   # case-insensitive lookup ('V', 'v', ...)
        if k.lower() == name:
            return d[k]
    return None


def curve_key(rpt):
    # The C/5 discharge curve: a dict of per-test V and Q lists, e.g. 'QV_discharge_C_5'
    keys = [k for k, v in rpt.items() if isinstance(v, dict) and 'discharge' in k.lower()
            and k.replace('/', '_').endswith('5') and field(v, 'v') is not None and field(v, 'q') is not None]
    return keys[0] if keys else None


def q_of_v(v, q, cap_ah):
    """Discharged capacity (Ah) on V_GRID from one C/5 discharge curve, or None if unusable."""
    n = min(len(v), len(q))
    v, q = np.asarray(v[:n], dtype=float), np.asarray(q[:n], dtype=float)
    ok = np.isfinite(v) & np.isfinite(q)
    v, q = v[ok], q[ok]
    if len(v) < 20:
        return None
    q = np.abs(q - q[0])                          # capacity discharged since the start of the discharge
    full = q.max()
    if not full > 0:
        return None
    f = min(UNIT_FACTORS, key=lambda u: abs(np.log(full * u / cap_ah)))   # unit that matches the C/5 capacity
    if not 0.7 < full * f / cap_ah < 1.3:
        return None
    q = q * f
    keep = v <= np.minimum.accumulate(v)          # monotone part: each point sets a new voltage minimum
    v, q = v[keep], q[keep]
    if v.min() > V_LO or v.max() < V_HI:
        return None
    order = np.argsort(v, kind='stable')
    return np.interp(V_GRID, v[order], q[order])


steps = pd.read_csv(OUTPUT)                       # rul_steps.csv from the cell above
need = steps.groupby('cell')['step'].apply(set).to_dict()
rpt_paths = json_paths('RPT_json')
cells = [c for c in need if c in rpt_paths]
print(f'Cells in rul_steps.csv: {len(need)} | RPT files found: {len(cells)}')

first = load_json(rpt_paths[cells[0]])
print('RPT keys of', cells[0], ':', list(first.keys()))
key = curve_key(first)
print('C/5 discharge curve key:', key, '| fields:', list(first[key].keys()) if key else None)
if key is None:
    raise SystemExit('No C/5 discharge curve found: send the key list above back to Claude.')
del first

COLS = ['c2_ratio', 'dq_mean', 'dq_min', 'dq_logvar', 'ica_peak_v', 'ica_peak_h', 'curve_max_ah']
rows, t0, no_curve, no_ref = [], time.time(), 0, 0
for n, cell in enumerate(cells):
    rpt = load_json(rpt_paths[cell])
    cap = to_num(rpt['capacity_discharge_C_5'])
    t_rpt = to_time(rpt['start_stop_time']['start'])
    c2_key = next((k for k in rpt if k.lower().startswith('capacity_discharge')
                   and k.replace('/', '_').endswith('2')), None)
    cap2 = to_num(rpt[c2_key]) if c2_key else None
    k = curve_key(rpt)
    vv, qq = (field(rpt[k], 'v'), field(rpt[k], 'q')) if k else ([], [])
    m = min(len(cap), len(t_rpt))
    valid = np.where((cap[:m].notna() & t_rpt[:m].notna()).values)[0]    # same tests as rul_rows()
    ref = None
    for step, i in enumerate(valid):
        if step not in need[cell]:
            continue
        row = {'cell': cell, 'step': step, **{c: np.nan for c in COLS}}
        if cap2 is not None and i < len(cap2) and np.isfinite(cap2.iloc[i]):
            row['c2_ratio'] = cap2.iloc[i] / cap.iloc[i]                    # rate capability: C/2 vs C/5
        curve = None
        try:
            if i < len(vv) and i < len(qq) and isinstance(vv[i], list) and isinstance(qq[i], list):
                curve = q_of_v(vv[i], qq[i], cap.iloc[i])
        except (TypeError, ValueError):
            curve = None
        if step == 0:
            ref = curve                                                      # ΔQ(V) is relative to the first test
        if curve is None:
            no_curve += 1
        else:
            row['curve_max_ah'] = float(curve.max())
            dqdv = -np.gradient(np.convolve(curve, np.ones(9) / 9, mode='same'), V_GRID)[8:-8]
            j = int(np.argmax(dqdv))
            row['ica_peak_v'], row['ica_peak_h'] = float(V_GRID[8:-8][j]), float(dqdv[j])
            if ref is not None:
                dq = curve - ref
                row.update(dq_mean=float(dq.mean()), dq_min=float(dq.min()),
                           dq_logvar=float(np.log10(dq.var() + 1e-12)) if step > 0 else np.nan)
            else:
                no_ref += 1
        rows.append(row)
    if n % 20 == 0 or n == len(cells) - 1:
        print(f'{n + 1}/{len(cells)} cells, {time.time() - t0:.0f} s')
    del rpt

dqv = pd.DataFrame(rows, columns=['cell', 'step'] + COLS)
dqv.to_csv(DQV_OUTPUT, index=False)
print(f'\nSaved {len(dqv)} rows for {dqv["cell"].nunique()} cells to {DQV_OUTPUT}')
print(f'Tests without a usable curve: {no_curve} | with a curve but no first-test reference: {no_ref}')

check = dqv.merge(steps[['cell', 'step', 'capacity']], on=['cell', 'step'])
print('Capacity down to 3.40 V / C/5 capacity:', (check['curve_max_ah'] / check['capacity']).describe().round(3).to_dict())